# Minimizing a Finite State Machine

In this notebook we show how a <span style="font-variant:small-caps;">Dfa</span> can be minimized by identifying states the are equivalent.

In [ ]:
#load "AutomataTypes.fsx"

In [ ]:
open AutomataTypes

## Some type definitions
The input is `Dfa<'state>` and the result is `Dfa<Set<'state>>`: each resulting state is an equivalence class. The generic state type supports comparison, making immutable sets, pairs and transition maps usable as keys. This algorithm requires a complete transition function; missing transitions must first be completed with a rejecting sink.

In [ ]:
let requireComplete (dfa: Dfa<'state>) =
    if not (Set.contains dfa.Start dfa.States) || not (Set.isSubset dfa.Accepting dfa.States) then
        invalidArg "dfa" "The start and accepting states must belong to the state set."
    for KeyValue((source, c), target) in dfa.Transitions do
        if not (Set.contains source dfa.States && Set.contains target dfa.States && Set.contains c dfa.Alphabet) then
            invalidArg "dfa" "Every transition must use declared states and alphabet symbols."
    for state in dfa.States do
        for c in dfa.Alphabet do
            if not (Map.containsKey (state, c) dfa.Transitions) then
                invalidArg "dfa" "This algorithm requires a complete DFA; add a rejecting sink for missing transitions."

The function `arb(M)` takes a non-empty set `M` as its argument and returns an *arbitrary* element from this set.  The set `M` is not changed.  All F# sets are immutable. 

F# infers the generic element type.

The function `cartesianProduct A B` computes the *Cartesian product* $A \times B$ of the sets $A$ and $B$ where $A \times B$ is defined as follows:
$$ A \times B := \{ (x, y) \mid x \in A \wedge y \in B \}. $$
In order to denote the type of the second set, we define the abstract type `T`.

In [ ]:
let arb states =
    if Set.isEmpty states then invalidArg "states" "Cannot choose an element of an empty set."
    Set.minElement states

let cartesianProduct first second =
    seq { for x in first do for y in second do yield x, y } |> Set.ofSeq

In [ ]:
cartesianProduct (set [1;2]) (set ['a';'b';'c'])

The function `separate` takes four arguments:
- `Pairs`  is a set of pairs of states from some given <span style="font-variant:small-caps;">Dfa</span> $F$.

   If $(p_1, p_2) \in \texttt{Pairs}$, then $p_1$ and $p_2$ are known to be *separable*.
   
   Two states $p_1$ and $p_2$ are *separable* if there is a string $w$ such that processing $w$ in state
   $p_1$ leads to an accepting state, while processing $w$ in state $p_2$ leads to a state that is not 
   accepting, or vice versa.
- `States` is the set of all states of the <span style="font-variant:small-caps;">Fsm</span> $F$,
- `Σ` is the alphabet of the <span style="font-variant:small-caps;">Dfa</span> $F$.
- `𝛿` is the transition function of the <span style="font-variant:small-caps;">Dfa</span>

The function `separate(Pairs, States, Σ, 𝛿)` computes the set of pairs of states $(q_1, q_2)$ that are *separable* because there is some character $c \in \Sigma$ such that 
$$\delta(q_1,c) = p_1, \quad \textrm{but} \quad \delta(q_2,c) = p_2 $$
and $(p_1, p_2)$ is already known to be *separable* because $(p_1, p_2) \in \textrm{Pairs}$.

Given a state `p` and a `Partition` of the set of all states, the function `findEquivalenceClass p partition` returns the equivalence class of `p`, i.e. it returns the set of states from `Partition` that contains the state `x`.

In [ ]:
let separate pairs states alphabet (delta: Map<'state * char, 'state>) =
    seq { for q1 in states do
              for q2 in states do
                  if alphabet |> Seq.exists (fun c -> Set.contains (delta[q1, c], delta[q2, c]) pairs) then
                      yield q1, q2 }
    |> Set.ofSeq

let findEquivalenceClass state partition =
    partition |> Seq.find (Set.contains state)

The function `allSeparable states accepting alphabet delta` takes four arguments:
* `Q` is the set of states of the Fsm.
* `A` is the set of all accepting states,
* `Σ`  is the alphabet.
* `𝛿` is the transition function.  

  `𝛿` is represented as a map.   

The function computes the set of all Pairs `(p, q)` such that `p` and `q` are separable, i.e. all pairs such that
$$ \exists s \in \Sigma^*: \bigl(\delta^*(p, s) \in A \wedge \delta^*(q,s) \not\in A\bigr) \vee 
                           \bigl(\delta^*(p, s) \not\in A \wedge \delta^*(q,s) \in A\bigr). 
$$
The result is computed via a *fixed point* computation:
1. Initially, all accepting states are separated from the non-accepting states.
2. If $p_1$ and $p_2$ are separable and $q_1$ and $q_2$ are two states such that there is a character 
   $c \in \Sigma$ such that
   $$ \delta(q_1, c) = p_1 \quad \mbox{and} \quad \delta(q_2, c) = p_2, $$
   then $q_1$ and $q_2$ are separable.

In [ ]:
let allSeparable states accepting alphabet delta =
    let accepting = Set.intersect states accepting
    let nonAccepting = Set.difference states accepting
    let initial = Set.union (cartesianProduct nonAccepting accepting) (cartesianProduct accepting nonAccepting)
    let rec expand pairs =
        let newPairs = separate pairs states alphabet delta
        if Set.isSubset newPairs pairs then pairs else expand (Set.union pairs newPairs)
    expand initial

The function `reachable start alphabet delta` takes three arguments:
* `q0` is the start state of an Fsm,
* `Σ`  is the alphabet.
* `𝛿`  is the transition function.  The transition function is assumed to be *complete*. `𝛿` is represented as a map.   

It returns the set of all states that can be reached from the start state `q0` by reading strings of characters from `Σ`.

The result is computed via a *fixed point* computation.

In [ ]:
let reachable start alphabet (delta: Map<'state * char, 'state>) =
    let rec expand states =
        let newStates =
            seq { for state in states do for c in alphabet do yield delta[state, c] } |> Set.ofSeq
        if Set.isSubset newStates states then states else expand (Set.union states newStates)
    expand (Set.singleton start)

The function `minimize(A)` takes a <span style="font-variant:small-caps;">Dfa</span> `F` as its input.
The record fields correspond to the five components
$$ F = (Q, \Sigma, \delta, q_0, A) $$
The algorithm performs the following steps:
1. All unreachable states are eliminated.
2. All accepting states are separated form all non-accepting states.
3. States are separated as long as possible.
   Two states $p_1$ and $p_2$ are separable if there is a string 
   $w \in \Sigma^*$ such that 
   $$ \Bigl(\delta(p_1,w) \in A \;\wedge\; \delta(p_2,w) \not\in A\Bigr) \quad \vee \quad 
      \Bigl(\delta(p_1,w) \not\in A \;\wedge\; \delta(p_2,w) \in A\Bigr)
   $$
4. States that are not separable are *equivalent* and are therefore identified and grouped
   in equivalence classes.  The states in an equivalence class are then identified.

`quotient` builds the automaton from its equivalence classes. It is shared with the fixed-point variant in `Minimize-FP.ipynb`. Accepting states outside the reachable set are removed before pair separation, and all retained classes are non-empty.

In [ ]:
let quotient (dfa: Dfa<'state>) states separable : Dfa<Set<'state>> =
    let equivalent = Set.difference (cartesianProduct states states) separable
    let classes = states |> Set.map (fun q -> states |> Set.filter (fun p -> Set.contains (p, q) equivalent))
    let classOf q = findEquivalenceClass q classes
    let transitions =
        seq { for source in states do
                  for c in dfa.Alphabet do
                      yield (classOf source, c), classOf dfa.Transitions[source, c] }
        |> Map.ofSeq
    { States = classes; Alphabet = dfa.Alphabet; Transitions = transitions
      Start = classOf dfa.Start
      Accepting = classes |> Set.filter (fun states -> Set.contains (arb states) dfa.Accepting) }

let minimize (dfa: Dfa<'state>) : Dfa<Set<'state>> =
    requireComplete dfa
    let states = reachable dfa.Start dfa.Alphabet dfa.Transitions
    let separable = allSeparable states dfa.Accepting dfa.Alphabet dfa.Transitions
    quotient dfa states separable